In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")


In [ ]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()


In [ ]:
df.show(10, truncate=False)

In [ ]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'


In [ ]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()


In [ ]:
from pyspark.sql.functions import min as _min, max as _max, sum as _sum

# Uzupełniony kod do Zadania 2.2
category_summary = (
    df.groupBy("category")
    .agg(
        _sum("amount").alias("suma_PLN"),
        _min("amount").alias("min_PLN"),
        _max("amount").alias("max_PLN")
    )
    .orderBy("category")
)

category_summary.show()


In [ ]:
from pyspark.sql.functions import min as _min, max as _max, sum as _sum

# Uzupełniony kod do Zadania 2.2
category_summary = (
    df.groupBy("category")
    .agg(
        _sum("amount").alias("suma_PLN"),
        _min("amount").alias("min_PLN"),
        _max("amount").alias("max_PLN")
    )
    .orderBy("category")
)

category_summary.show()


In [ ]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)


In [ ]:
window_30m_store = (
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("*").alias("liczba_tx"),
        _sum("amount").alias("suma_PLN")
    )
    .orderBy("window", "store")
)

window_30m_store.show(truncate=False)


In [ ]:
from pyspark.sql.functions import desc


krakow_max_revenue_hour = (
    df.filter(df.store == "Kraków")                     
    .groupBy(window("timestamp", "1 hour"))           
    .agg(
        _sum("amount").alias("suma_PLN")                 
    )
    .orderBy(desc("suma_PLN"))                           
)

krakow_max_revenue_hour.show(truncate=False)


In [ ]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)


In [ ]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")


In [ ]:
# Okna sliding (przesuwne) dają więcej wierszy wynikowych, ponieważ nakładają się na siebie. 
# W podejściu tumbling okno 1-godzinne tworzy się dokładnie co godzinę (np. 8:00-9:00, 9:00-10:00). 
# Natomiast w oknie sliding z krokiem 30 minut, nowe okno startuje dwa razy częściej: co pół godziny 
# (np. 8:00-9:00, 8:30-9:30, 9:00-10:00). Na tej samej przestrzeni czasu powstaje po prostu 
# więcej przedziałów (okien), a pojedyncze transakcje wpadają do kilku okien jednocześnie 
# (np. transakcja z 08:45 wpadnie i do okna 8:00-9:00, i do 8:30-9:30).


In [ ]:
# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Odp: 4661
# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    Odp: groupBy("store") bierze wszystkie dane jak leci i zwraca dla każdego 
#               sklepu JEDEN wiersz z łącznym wynikiem z całego dnia. 
#               groupBy(window(...), "store") tnie najpierw czas na okna, a potem w 
#               każdym z nich grupuje sklepy. Dzięki temu jeden sklep ma wiele wierszy 
#               po jednym dla każdego przedziału czasowego, w którym coś sprzedał.

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Odp: Transakcja z godziny 09:30 znajdzie się dokładnie w 2 oknach.
#               Okna tworzą się co 30 minut i trwają godzinę, więc mamy:
#               1) [08:30 - 09:30] -> w sparku czas końcowy jest wykluczony, więc ułamki sekund po 9:30 tu nie wejdą.
#               2) [09:00 - 10:00] -> zawier 09:30
#               3) [09:30 - 10:30] -> zawiera 09:30


In [ ]:
from pyspark.sql.functions import avg, asc, desc

# Praca domowa
# Zad 1
gdansk_lowest_avg = (
    df.filter(df.store == "Gdańsk")                 
    .groupBy(window("timestamp", "1 hour"))         
    .agg(avg("amount").alias("srednia_PLN"))            
    .orderBy(asc("srednia_PLN"))                         
)
gdansk_lowest_avg.show(1, truncate=False)                


# Zad 2
print("--- Praca domowa 2: Kategorie w oknie 09:00-09:30 ---")
kategorie_0900 = (
    df.groupBy(window("timestamp", "30 minutes"), "category")
    .count()
    .filter(col("window.start").cast("string").like("%09:00:00"))
    .orderBy("category")
)
kategorie_0900.show(truncate=False)


#Zad3
print("--- Praca domowa 3: Szczyt w oknach 15-minutowych ---")
szczyt_15m = (
    df.groupBy(window("timestamp", "15 minutes"))       
    .count()                                             
    .orderBy(desc("count"))                              
)
szczyt_15m.show(1, truncate=False)                     


spark.stop()
print("Sesja Spark została poprawnie zakończona!")
